<h1 style="text-align: center;">Physical AI의 Vision-LLM 융합 시청각 멀티모달 시스템</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
김규래&emsp;<br>
kkr.kyurae.kim@gmail.com&emsp;
</div><br>

---
---

## 4. 딥러닝 기반 Edge AI 객체 탐지 시스템

객체 탐지(Object Detection)란?<br>
"이미지나 영상에서 객체의 위치와 객체의 종류를 찾아내는 컴퓨터 비전 기술"

* 이미지 또는 영상 속 객체를 자동으로 탐지
* 객체의 위치(Bounding Box)와 종류(Class)를 예측
* 위치 추정(Localization)과 이미지 분류(Image Classification)를 결합한 기술
* 하나의 이미지에서 여러 객체를 동시에 인식 가능

Rule-based vs. Learning-based:
* 규칙기반 (Rule-based) 객체 탐지
  * 사람이 색상, 크기, 형태 등의 규칙을 직접 설정
  * 학습 데이터가 필요하지 않음
  * 환경이 달라지면 규칙을 다시 수정해야 함
  * 조명, 배경, 객체의 크기와 방향 변화에 민감함
* 학습기반 (Learning-based) 객체 탐지
  * 객체의 특징을 모델이 데이터로부터 학습
  * 많은 학습 데이터와 연산이 필요
  * 복잡한 배경과 다양한 객체 변화에 대응 가능
  * 내부 판단 과정을 직접 해석하기 어려움

---

### A. Bounding Box

Bounding Box (BBox)는 이미지나 영상의 프레임에서 탐지된 객체를 둘러싸는 직사각형 영역입니다.

기본적으로 대부분의 객체 탐지는 OpenCV 프레임워크에 기반을 두고있기에 OpenCV 좌표계 시스템을 사용합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import skimage.data

In [ ]:
image = skimage.data.astronaut()
plt.imshow(image)

h, w, c = image.shape

# 좌측 상단
circle0 = Circle((0, 0), radius=25, edgecolor='magenta', facecolor='magenta', linewidth=5)
plt.gca().add_patch(circle0)

# 우측 하단
circle1 = Circle((w, h), radius=25, edgecolor='cyan', facecolor='cyan', linewidth=5)
plt.gca().add_patch(circle1)

print(f"이미지 높이: {h}\n이미지 너비: {w}")

BBox의 표현 방식은 크게 네 가지가 있습니다:
1. (x1, y1, x2, y2)
2. (x, y, w, h)
3. (cx, cy, w, h)
4. (cx_norm, cy_norm, w_norm, h_norm)

각 표현 방식을 살펴보겠습니다.

1. `bbox = (x1, y1, x2, y2)`
    * `(x1, y1)`: 좌측 상단 좌표
    * `(x2, y2)`: 우측 하단 좌표
    * bbox의 너비: `x2 - x1`
    * bbox의 높이: `y2 - y1`
    * OpenCV로 박스를 그릴 때 편리: `cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)`

2. `bbox = (x, y, w, h)`
    * `(x, y)`: 좌측 상단 좌표
    * `(w, h)`: bbox 크기
    * 우측 하단 좌표: `(x + w, y + h)`
    * OpenCV 일부 함수에서 이 방식을 사용 (`cv2.selectROI`, `cv2.boundingRect` 등)

3. `bbox = (cx, cy, w, h)`
    * `(cx, cy)`: bbox 중심 좌표
    * `(w, h)`: bbox 크기
    * 좌측 상단 좌표: `(int(cx - w/2), int(cy - h/2))`
    * 우측 하단 좌표: `(int(cx + w/2), int(cy + h/2))`

4. `bbox = (cx_norm, cy_norm, w_norm, h_norm)`
    * `(cx, cy, w, h)`를 이미지 크기에 맞춰 0~1로 정규화
    * `cx_norm = cx / image_width`
    * `cy_norm = cy / image_height`
    * `w_norm = w / image_width`
    * `h_norm = h / image_height`

간단한 `cv2.rectangle`을 사용해 1번 방식인 `(x1, y1, x2, y2)` BBox를 그려봅시다.

In [ ]:
cup_img = skimage.data.coffee()

x1, y1 = 170, 16
x2, y2 = 410, 330

cv2.rectangle(cup_img, (x1, y1), (x2, y2), (0, 255, 0), 3)
cv2.putText(cup_img, "Cup\n0.95", (x1-60, y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

plt.imshow(cup_img);

이 방식은 하드코딩으로 좌표를 직접 입력한 방식이기에 진정한 객체 탐지라고 볼 수 없습니다. 이번에는 직접 좌표를 설정하지 않고 BBox를 그리는 실제 객체 탐지를 구현해봅시다.

가장 간단한 규칙기반 객체 탐지 방법으로는 Color Segmentation이 있습니다. 사람이 원하는 객체에 맞춰 직접 색상 범위를 지정하여 탐지합니다. Color Segmentation의 결과로 Binary Mask가 생성이 되고, 마스크를 통해 Contour를 추출할 수 있습니다.

규칙기반 객체 탐지로 BBox를 그리기 위해서는 Contour가 필요합니다.

우선, 이전 Computer Vision 실습에서 연습한 방식과 동일하게 Color Segmentation을 통해 Contour를 추출해봅시다.

In [ ]:
apple = cv2.imread("src/images/apple.png")
apple_rgb = cv2.cvtColor(apple, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16,8))
plt.imshow(apple_rgb)
plt.axis("off");

In [ ]:
# TODO: Color Segmentation을 통해 Binary Mask 생성

# 사과 빨강 HSV 범위
red_lower1 = np.array([0, 100, 50])
red_upper1 = np.array([10, 255, 255])
red_lower2 = np.array([170, 100, 50])
red_upper2 = np.array([180, 255, 255])

# TODO: Gaussian Blur 적용
apple_blur = ...

# TODO: Binary Mask 생성
apple_mask = ...


# 이미지 출력
plt.figure(figsize=(16,8))

plt.subplot(1,2,1), plt.imshow(apple_rgb), plt.title("Original Image")
plt.subplot(1,2,2), plt.imshow(apple_mask, cmap="gray"), plt.title("Binary Mask")

for ax in plt.gcf().axes:
    ax.axis("off")

In [ ]:
# TODO: Binary Mask를 사용하여 Contour 추출

# TODO: 필요하다면 Morphological Operation 적용
apple_mask = ...

# TODO: Contour를 추출하여 원본 이미지 위에 overlay
final_img = ...
contours, _ = ...
if len(contours) > 0:
    # TODO: 가장 큰 contour 선택
    largest_contour = ...

    # 가장 큰 contour만 채운 binary mask 생성
    largest_mask = np.zeros_like(apple_mask)
    cv2.drawContours(largest_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = largest_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

plt.figure(figsize=(16,8))
plt.imshow(final_img)
plt.axis("off")

추출한 Contour를 사용해 객체 탐지를 할 수 있습니다.

`cv2.boundingRect()` 함수를 사용하면 이미지에서 찾은 Contour를 감싸는 최소 사각형을 자동으로 계산합니다.

BBox 표현은 2번 방식인 "`bbox = (x, y, w, h)`"으로 표현됩니다.

In [ ]:
x, y, w, h = cv2.boundingRect(largest_contour)
print(x, y, w, h)

이 값을 사용하여 원본 이미지 위에 BBox를 그려봅시다.

In [ ]:
def draw_bbox(img, coord1, coord2, color, thickness=3, txt=""):
    _, txt_h = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
    cv2.rectangle(img, coord1, coord2, color, thickness)
    cv2.putText(img, txt, (coord1[0], coord1[1]-txt_h+15), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

In [ ]:
apple = cv2.imread("src/images/apple.png")
apple_rgb = cv2.cvtColor(apple, cv2.COLOR_BGR2RGB)

# TODO: cv2.rectangle() 함수에 사용할 수 있는 좌표로 변환
x1, y1 = ...
x2, y2 = ...

draw_bbox(apple_rgb, (x1, y1), (x2, y2), (0, 255, 0), txt="Apple, 0.98")

plt.figure(figsize=(16,8))
plt.imshow(apple_rgb);

이로써 BBox의 좌표를 하드코딩하는 것이 아닌 규칙기반 객체 탐지인 Color Segmentation으로 BBox를 그려봤습니다.

---

### B. Intersection over Union (IoU)

Intersection over Union (IoU):
* 두 Bounding Box가 얼마나 겹치는지를 나타내는 값
* (교집합 면적) / (합집합 면적)
* 값의 범위: 0 ≤ IoU ≤ 1
  * IoU = 0:  전혀 겹치지 않음
  * IoU = 1:  두 BBox가 완전히 동일


```python
bbox_a = (xa_1, ya_1, xa_2, ya_2)
bbox_b = (xb_1, yb_1, xb_2, yb_2)
```

두개의 BBox를 위와 같이 표현한다면, 좌표들을 사용해 교집합 박스의 좌표를 구할 수 있습니다.

```python
intersection_x1 = max(xa_1, xb_1)
intersection_y1 = max(ya_1, yb_1)
intersection_x2 = min(xa_2, xb_2)
intersection_y2 = min(ya_2, yb_2)
```

<br>
그 다음, 교집합 박스의 너비와 높이를 계산합니다.

```python
w = max(0, intersection_x2 - intersection_x1)
h = max(0, intersection_y2 - intersection_y1)
```

겹치는 영역이 없다면 `intersection2 - intersection1`의 값이 음수가 되어 너비와 높이가 0이 됩니다.

<br>
교집합의 넓이를 계산했다면, 최종적으로 합집합의 넓이까지 계산하여 IoU 값을 구할 수 있습니다.

IoU 계산 함수를 구현해봅시다.

In [ ]:
# TODO: 1) BBox 좌표를 (x,y,w,h)에서 (x1,y1,x2,y2)로 변환하는 함수 정의
# TODO: 2) IoU 계산 함수 정의

def convert_bbox_coord(bbox):
    pass

def calculate_iou(bbox_a, bbox_b):
    intersection_area = ...
    union_area = ...

    # TODO: ZeroDivisionError 주의
    return intersection_area / union_area

이제 3개의 이미지에서 IoU를 계산해봅시다.

In [ ]:
apples1 = cv2.imread("src/images/two_apples1.png")
apples2 = cv2.imread("src/images/two_apples2.png")
apples3 = cv2.imread("src/images/two_apples3.png")
apples1_rgb = cv2.cvtColor(apples1, cv2.COLOR_BGR2RGB)
apples2_rgb = cv2.cvtColor(apples2, cv2.COLOR_BGR2RGB)
apples3_rgb = cv2.cvtColor(apples3, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16,6))

plt.subplot(1,3,1), plt.imshow(apples1_rgb)
plt.subplot(1,3,2), plt.imshow(apples2_rgb)
plt.subplot(1,3,3), plt.imshow(apples3_rgb)

for ax in plt.gcf().axes:
    ax.axis("off")

In [ ]:
def get_contour(img_bgr):
    red_lower1 = np.array([0, 100, 50])
    red_upper1 = np.array([10, 255, 255])
    red_lower2 = np.array([170, 100, 50])
    red_upper2 = np.array([180, 255, 255])
    green_lower = np.array([25, 10, 30])
    green_upper = np.array([100, 255, 255])

    img_blr = cv2.GaussianBlur(img_bgr, (7, 7), 0)
    blr_hsv = cv2.cvtColor(img_blr, cv2.COLOR_BGR2HSV)

    red_mask1 = cv2.inRange(blr_hsv, red_lower1, red_upper1)
    red_mask2 = cv2.inRange(blr_hsv, red_lower2, red_upper2)
    red_mask = cv2.bitwise_or(red_mask1, red_mask2)
    grn_mask = cv2.inRange(blr_hsv, green_lower, green_upper)

    contours_red, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_grn, _ = cv2.findContours(grn_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cntr_red, cntr_grn = None, None

    if len(contours_red) > 0:
        cntr_red = max(contours_red, key=cv2.contourArea)
    if len(contours_grn) > 0:
        cntr_grn = max(contours_grn, key=cv2.contourArea)

    return cntr_red, cntr_grn

In [ ]:
apples1 = cv2.imread("src/images/two_apples1.png")
apples2 = cv2.imread("src/images/two_apples2.png")
apples3 = cv2.imread("src/images/two_apples3.png")

apples1_rgb = cv2.cvtColor(apples1, cv2.COLOR_BGR2RGB)
apples2_rgb = cv2.cvtColor(apples2, cv2.COLOR_BGR2RGB)
apples3_rgb = cv2.cvtColor(apples3, cv2.COLOR_BGR2RGB)

cntr_red1, cntr_grn1 = get_contour(apples1)
cntr_red2, cntr_grn2 = get_contour(apples2)
cntr_red3, cntr_grn3 = get_contour(apples3)

In [ ]:
bbox1 = cv2.boundingRect(cntr_red1)
bbox2 = cv2.boundingRect(cntr_grn1)

x_r, y_r, w_r, h_r = bbox1
x_g, y_g, w_g, h_g = bbox2

x_r1, y_r1 = x_r, y_r
x_r2, y_r2 = x_r+w_r, y_r+h_r
x_g1, y_g1 = x_g, y_g
x_g2, y_g2 = x_g+w_g, y_g+h_g

draw_bbox(apples1_rgb, (x_r1, y_r1), (x_r2, y_r2), (255, 0, 0), txt="Red Apple")
draw_bbox(apples1_rgb, (x_g1, y_g1), (x_g2, y_g2), (0, 255, 0), txt="Green Apple")

plt.figure(figsize=(12,8))
plt.imshow(apples1_rgb);

In [ ]:
iou1 = calculate_iou(convert_bbox_coord(bbox1), convert_bbox_coord(bbox2))
print(iou1)

In [ ]:
bbox1 = cv2.boundingRect(cntr_red2)
bbox2 = cv2.boundingRect(cntr_grn2)

x_r, y_r, w_r, h_r = bbox1
x_g, y_g, w_g, h_g = bbox2

x_r1, y_r1 = x_r, y_r
x_r2, y_r2 = x_r+w_r, y_r+h_r
x_g1, y_g1 = x_g, y_g
x_g2, y_g2 = x_g+w_g, y_g+h_g

draw_bbox(apples2_rgb, (x_r1, y_r1), (x_r2, y_r2), (255, 0, 0), txt="Red Apple")
draw_bbox(apples2_rgb, (x_g1, y_g1), (x_g2, y_g2), (0, 255, 0), txt="Green Apple")

plt.figure(figsize=(12,8))
plt.imshow(apples2_rgb);

In [ ]:
iou2 = calculate_iou(convert_bbox_coord(bbox1), convert_bbox_coord(bbox2))
print(iou2)

In [ ]:
bbox1 = cv2.boundingRect(cntr_red3)
bbox2 = cv2.boundingRect(cntr_grn3)

x_r, y_r, w_r, h_r = bbox1
x_g, y_g, w_g, h_g = bbox2

x_r1, y_r1 = x_r, y_r
x_r2, y_r2 = x_r+w_r, y_r+h_r
x_g1, y_g1 = x_g, y_g
x_g2, y_g2 = x_g+w_g, y_g+h_g

draw_bbox(apples3_rgb, (x_r1, y_r1), (x_r2, y_r2), (255, 0, 0), txt="Red Apple")
draw_bbox(apples3_rgb, (x_g1, y_g1), (x_g2, y_g2), (0, 255, 0), txt="Green Apple")

plt.figure(figsize=(12,8))
plt.imshow(apples3_rgb);

In [ ]:
iou3 = calculate_iou(convert_bbox_coord(bbox1), convert_bbox_coord(bbox2))
print(iou3)

이제 3가지 이미지의 IoU를 비교해봅시다.

In [ ]:
plt.figure(figsize=(16,6))

plt.subplot(1,3,1), plt.imshow(apples1_rgb), plt.title(f"IoU: {iou1:.4f}")
plt.subplot(1,3,2), plt.imshow(apples2_rgb), plt.title(f"IoU: {iou2:.4f}")
plt.subplot(1,3,3), plt.imshow(apples3_rgb), plt.title(f"IoU: {iou3:.4f}")

for ax in plt.gcf().axes:
    ax.axis("off")

---

### C. Non-Maximum Suppression (NMS)

비최대 억제 (NMS):
* 동일한 객체에 대해 여러 개의 Bounding Box가 높은 신뢰도로 예측될 수 있음
* 중복 Bounding Box를 제거하는 알고리즘
* Confidence Score와 IoU 기준으로 Bounding Box 선택
* 중복 제거 과정을 반복하여 객체마다 하나의 대표 Bounding Box만 남김

NMS 과정 (2~6번):
1. 신뢰도가 너무 낮은 BBox 제거 (Confidence Thresholding)
2. Confidence가 가장 높은 BBox 선택
3. 선택한 BBox와 다른 박스들의 IoU 계산
4. IoU가 임계값보다 높은 박스 제거 (IoU Thresholding)
5. 다음으로 Confidence가 가장 높은 BBox 선택
6. 3, 4, 5번 반복


NMS 알고리즘을 구현하기 앞서, 우선 랜덤으로 BBox를 생성해봅시다.

In [ ]:
from random import randint, gauss

def random_bbox(img, min_w=100, min_h=400, max_w=600, max_h=600):
    img_h, img_w = img.shape[:2]

    w = randint(min_w, min(max_w, img_w))
    h = randint(min_h, min(max_h, img_h))

    x1 = randint(0, img_w - w)
    y1 = randint(0, img_h - h)

    return x1, y1, x1 + w, y1 + h

def random_conf():
    return min(1.0, max(0.0, gauss(0.6, 0.1)))

In [ ]:
hubble = skimage.data.hubble_deep_field()

N = 20

boxes = []
scores = []

for i in range(N):
    boxes.append(random_bbox(hubble))
    scores.append(random_conf())

for i, box in enumerate(boxes):
    draw_bbox(
        hubble,
        (box[0], box[1]),
        (box[2], box[3]),
        (randint(0,255), randint(0,255), randint(0,255)),
        txt=f"{scores[i]:.2f}"
    )

plt.figure(figsize=(10,8))
plt.imshow(hubble)
plt.axis("off");

이제 이 난잡한 BBox 정글을 NMS로 정리해보도록 합시다.

In [ ]:
# TODO: Confidence Thresholding 함수 정의
# TODO: Confidence Score가 특정 임계값보다 낮으면 해당 BBox 제거

def prune_low_conf(boxes, scores, conf_threshold=0.5):
    """
    Confidence Thresholding 적용

    Parameters
    ----------
    boxes : list
        Bounding Box 목록
        각 박스는 (x1, y1, x2, y2) 형식

    scores : list
        각 Bounding Box의 Confidence Score

    conf_threshold : float
        Bounding Box를 유지하기 위한 최소 Confidence Score

    Returns
    -------
    pruned_boxes : list
        Confidence Score가 임계값 이상인 Bounding Box 목록

    pruned_scores : list
        Confidence Score가 임계값 이상인 Bounding Box의 Confidence Score 목록
    """

    pass

In [ ]:
# TODO: NMS 함수 정의
# TODO: np.sort() 함수를 사용하면 1차원 배열을 정렬
# TODO: np.argsort() 함수를 사용하면 1차원 배열의 정렬된 인덱스를 반환

def apply_nms(boxes, scores, iou_threshold=0.5):
    """
    NMS 적용

    Parameters
    ----------
    boxes : list
        Bounding Box 목록
        각 박스는 (x1, y1, x2, y2) 형식

    scores : list
        각 Bounding Box의 Confidence Score

    iou_threshold : float
        중복 Bounding Box를 제거할 IoU 기준값

    Returns
    -------
    nms_boxes : list
        NMS 적용 후 유지된 Bounding Box 목록

    nms_scores : list
        NMS 적용 후 유지된 Bounding Box의 Confidence Score 목록
    """

    pass

이제 Confidence Thresholding과 NMS를 적용해봅시다.

In [ ]:
print(len(boxes), len(scores))

new_boxes, new_scores = prune_low_conf(boxes, scores)
new_boxes, new_scores = apply_nms(boxes, scores, iou_threshold=0.2)

print(len(new_boxes), len(new_scores))

In [ ]:
hubble = skimage.data.hubble_deep_field()

for i, box in enumerate(new_boxes):
    draw_bbox(
        hubble,
        (box[0], box[1]),
        (box[2], box[3]),
        (randint(0,255), randint(0,255), randint(0,255)),
        txt=f"{scores[i]:.2f}"
    )

plt.figure(figsize=(10,8))
plt.imshow(hubble)
plt.axis("off");

확실히 중복된 Bounding Box는 제거된 모습을 확인할 수 있습니다.

여전히 중복되어 있는 BBox들은, 겹친 영역의 면적이 작거나 합집합의 면적이 넓어 교집합:합집합 비율이 임계값보다 낮은 경우입니다.

지금까지 NMS 알고리즘의 동작 원리를 이해하기 위해 직접 코드로 구현해 보았습니다.

OpenCV에서도 동일한 역할을 하는 내장 함수 `cv2.dnn.NMSBoxes()`가 제공됩니다.

OpenCV 함수를 사용해 같은 과정을 반복해 보고 결과를 비교해봅시다.

In [ ]:
def apply_nms_cv2(boxes, scores, conf_threshold=0.5, iou_threshold=0.5):
    """
    OpenCV의 cv2.dnn.NMSBoxes()를 이용해 NMS 적용

    Parameters
    ----------
    boxes : list
        Bounding Box 목록
        각 박스는 (x1, y1, x2, y2) 형식

    scores : list
        각 Bounding Box의 Confidence Score

    conf_threshold : float
        Bounding Box를 유지하기 위한 최소 Confidence Score

    iou_threshold : float
        중복 Bounding Box를 제거할 IoU 기준값

    Returns
    -------
    nms_boxes : list
        NMS 적용 후 유지된 Bounding Box 목록
        각 박스는 기존과 동일한 (x1, y1, x2, y2) 형식

    nms_scores : list
        NMS 적용 후 유지된 Bounding Box의 Confidence Score
    """

    if len(boxes) == 0:
        return [], []

    # cv2.dnn.NMSBoxes()는 (x, y, width, height) 형식을 사용
    boxes_xywh = []

    for box in boxes:
        x1, y1, x2, y2 = box

        width = x2 - x1
        height = y2 - y1

        boxes_xywh.append([int(x1), int(y1), int(width), int(height)])

    # OpenCV NMS 적용하여 BBox 제거 후 남은 BBox의 인덱스
    keep_indices = cv2.dnn.NMSBoxes(
        bboxes=boxes_xywh,
        scores=scores,
        score_threshold=conf_threshold,
        nms_threshold=iou_threshold,
    )

    # 1차원 리스트로 변환
    keep_indices = np.array(keep_indices).reshape(-1).tolist()

    # 원래 xyxy 형식의 박스에서 선택
    nms_boxes = [boxes[index] for index in keep_indices]
    nms_scores = [scores[index] for index in keep_indices]

    return nms_boxes, nms_scores

다시 NMS 적용 전 이미지를 확인합시다.

In [ ]:
hubble = skimage.data.hubble_deep_field()

for i, box in enumerate(boxes):
    draw_bbox(
        hubble,
        (box[0], box[1]),
        (box[2], box[3]),
        (randint(0,255), randint(0,255), randint(0,255)),
        txt=f"{scores[i]:.2f}"
    )

plt.figure(figsize=(10,8))
plt.imshow(hubble)
plt.axis("off");

OpenCV의 NMS 적용:

In [ ]:
print(len(boxes), len(scores))

cv2_new_boxes, cv2_new_scores = apply_nms_cv2(boxes, scores, iou_threshold=0.2)

print(len(cv2_new_boxes), len(cv2_new_boxes))

In [ ]:
hubble_cv2_nms = skimage.data.hubble_deep_field()

for i, box in enumerate(cv2_new_boxes):
    draw_bbox(
        hubble_cv2_nms,
        (box[0], box[1]),
        (box[2], box[3]),
        (randint(0,255), randint(0,255), randint(0,255)),
        txt=f"{scores[i]:.2f}"
    )

plt.figure(figsize=(10,8))
plt.imshow(hubble_cv2_nms)
plt.axis("off");

직접 구현한 NMS와 OpenCV 내장 NMS의 결과를 비교해봅시다.

In [ ]:
plt.figure(figsize=(16,10))

plt.subplot(1,2,1), plt.imshow(hubble), plt.title("Custom NMS Result")
plt.subplot(1,2,2), plt.imshow(hubble_cv2_nms), plt.title("CV2 NMS Result")

for ax in plt.gcf().axes:
    ax.axis("off")

직접 구현한 결과와 정확히 일치하는 것을 확인할 수 있습니다.

OpenCV 내장 함수를 사용하면 간단하게 NMS를 적용할 수 있지만, 알고리즘을 직접 구현해 봄으로써 그 내부 동작 원리를 깊이 있게 이해할 수 있습니다.

---

### D. YOLO 개요

지금까지 객체 탐지 시스템의 기초 원리와 주요 기법들을 살펴보았습니다.

이제 카메라 피드 기반의 실시간 객체 탐지를 위해, YOLO 모델을 알아보겠습니다.

**YOLO (You Only Look Once):**
* CNN 기반 객체 탐지 모델
* One-stage Detector
* 이미지 전체를 신경망에 한 번 입력하여, 이미지 속 객체들을 전부 탐지
* 객체들의 정보 동시 예측
  * 위치: Bounding Box
  * 종류: Class
  * 신뢰도: Confidence
* 실시간 처리에 적합
  * 별도의 Region Proposal 단계가 없음
  * Feature Map을 여러 객체가 공유함
  * 여러 위치의 예측을 병렬로 계산함
  * GPU의 병렬 연산을 효율적으로 활용
  * 작은 모델부터 큰 모델까지 선택 가능

우선, YOLO 모델을 활용하기 위해 공식 파이썬 라이브러리인 `ultralytics`를 설치해야합니다.

`ultralytics` 패키지를 설치하기 위해서는 공식 문서에 명시된 `numpy>=1.23.0`, `opencv-python>=4.6.0` 버전 요구사항에 맞춰야 하므로, 설치 전 현재 버전을 먼저 확인해 보겠습니다.

In [ ]:
import numpy as np
import cv2

print(np.__version__)
print(cv2.__version__)

확인 결과 `OpenCV`는 버전 요구사항에 맞지만, `NumPy`는 업그레이드가 필요합니다.

따라서 `NumPy`만 권장 버전으로 업그레이드를 진행해 보겠습니다.

주의사항으로는 `NumPy`를 업그레이드할 때 버전 2.0 이상으로 올라가지 않도록 주의해야 합니다. `NumPy 2.x` 버전에서는 GStreamer가 연동된 `OpenCV`와 호환성 문제가 발생하여 정상적으로 동작하지 않습니다.

```bash
pip install "numpy==1.23.0"
```

In [ ]:
import numpy as np
import cv2

print(np.__version__)
print(cv2.__version__)

위 코드에서 `NumPy` 버전이 업데이트 되지 않는다면, "Ctrl+Shift+P"을 눌러 "Jupyter: Restart Kernel"을 선택하여 다시 실행해봅시다.

다음으로, `ultralytics`를 의존성 패키지(Dependency) 없이 최소 설치 한 후, 필수 의존 패키지만 따로 추가 설치합시다.

```bash
pip install --no-deps ultralytics
pip install --no-deps "typing_extensions>=4.10.0,<5"
pip install --no-deps "filelock>=3.16.1,<4"
```

정상적으로 설치가 되었는지 아래 코드로 확인해봅시다.

In [ ]:
from ultralytics import YOLO

에러 없이 `ultralytics` 라이브러리가 import 되었다면 다음으로 진행하면 되겠습니다.

#### $1)$ YOLO 모델 선택

YOLO는 사용 목적과 컴퓨터 환경에 맞게 선택할 수 있도록 매우 다양한 모델 라인업을 제공합니다.

YOLO 모델의 이름은 크게 ***모델명 + 버전 + 모델 크기***의 구조로 이루어져 있습니다.

예) YOLO11n
* **YOLO**: **모델 이름** *(You Only Look Once)*
* **숫자 (11)**: **버전 정보** (V1부터 지속적으로 발전하여 최근에는 **YOLO11**까지 출시)
* **알파벳 (n)**: **모델 크기 및 파라미터 수** (n, s, m, l, x 등)

모델 크기(알파벳) 구분

| 알파벳 | 약자 | 특징 |
| :---: | :---: | :--- |
| **n** | Nano | 경량화 모델 (속도 최우선, 연산량 적음) |
| **s** | Small | 소형 모델 (가벼우면서 적절한 성능) |
| **m** | Medium | 중형 모델 (속도와 정확도의 균형) |
| **l** | Large | 대형 모델 (높은 정확도) |
| **x** | Extra Large | 초대형 모델 (최고 성능, 높은 연산량 필요) |

YOLO는 최신 버전인 `YOLO26`까지 출시되어 있지만, 본 수업에서는 아래와 같은 이유로 `YOLO11n` 모델을 선택하여 진행합니다.

1. **안정성과 호환성**: 최신 `YOLO26`에 비해 `YOLO11`은 라이브러리 및 실행 환경(PyTorch, OpenCV 등)과의 의존성 충돌 위험이 적고 매우 안정적
2. **실시간 가벼운 동작**: `n(Nano)` 모델은 연산량이 적어 한계가 많은 Jetson과 같은 엣지 디바이스에서도 실시간 카메라 피드 탐지를 무리 없이 수행
3. **빠르고 효율적인 실습**: 모델 가중치 파일 용량이 가벼워 로딩이 빠르며, 경량화 환경에서도 YOLO의 핵심 메커니즘을 학습하기에 최적화된 모델

"YOLO11n을 포함한 기본 YOLO 모델들은 수많은 이미지 데이터로 이미 사전 학습된 모델(Pre-trained Model)입니다.

이 모델은 일상적인 80가지 범주의 객체가 포함된 COCO(Common Objects in Context) 데이터셋으로 학습되어 있어, 별도의 추가 학습 없이도 바로 사람, 차량, 동물 등을 탐지할 수 있습니다.

#### $2)$ COCO 데이터셋

COCO 데이터셋:
* **컴퓨터 비전 대표 데이터셋**
  * Microsoft에서 제작한 대규모 객체 탐지(Object Detection) 및 분할(Segmentation) 데이터셋
* **80가지 클래스 제공**
  * 사람, 자동차, 강아지, 의자 등 일상에서 흔히 접하는 80가지 범주의 객체 정보가 포함되어 높은 활용도
* **복잡한 배경과 맥락(Context)**
  * 단일 물체만 깔끔하게 있는 이미지뿐만 아니라, 여러 객체가 어우러진 실제 환경 이미지가 다수 포함되어 있어 높은 실전 인식 성능

COCO 데이터셋의 80가지 클래스:

![](src/images/COCO_80_classes.png)

#### $3)$ YOLO 모델 최초 설치 및 불러오기

이제 `YOLO11n` 모델을 불러와 보겠습니다.

앞서 다룬 MNIST 및 CIFAR-10 데이터셋처럼, 파일을 지정한 경로에서 불러오거나 파일이 존재하지 않는다면 자동으로 다운로드됩니다.

최초 사용하는 모델이므로, 경로를 지정하여 설치해봅시다.

In [ ]:
model = YOLO("src/models/YOLO/yolo11n.pt")

Ultralytics는 GPU(CUDA)가 사용 가능한 환경이라면 기본적으로 모델을 `cuda:0` 디바이스에 자동으로 할당합니다.

만약 모델 실행 장치를 명시적으로 지정하고 싶다면 아래와 같이 디바이스 설정 코드를 추가해 줄 수 있습니다.

In [ ]:
model.to("cuda")

print(model.device)

#### $4)$ 정적 이미지에서의 YOLO 객체 탐지

이제 불러온 모델에 샘플 이미지를 적용하여 객체 탐지가 잘 동작하는지 확인해 볼까요?

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
results = model.predict(
    source="src/images/city.png",
    conf=0.25,   # Confidence Threshold
    iou=0.5,     # IoU Threshold
    classes=None,
)

yolo_test_img = results[0].plot()  # 일반 .py 파일에서는 results[0].show()

yolo_test_img_rgb = cv2.cvtColor(yolo_test_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,8))
plt.imshow(yolo_test_img_rgb)
plt.axis("off");

위 코드에서 `results[0]`와 같이 인덱싱을 한 이유는, YOLO가 여러 이미지를 동시에 처리(Batch processing)할 수 있어 결과가 리스트 형태로 반환되기 때문입니다.

In [ ]:
results = model.predict(
    source=[
        "src/images/city.png",
        "src/images/apple.png",
    ],
    conf=0.25,   # Confidence Threshold
    iou=0.5,     # IoU Threshold
    classes=None,
)

yolo_test_img0 = results[0].plot()
yolo_test_img1 = results[1].plot()

yolo_test_img0_rgb = cv2.cvtColor(yolo_test_img0, cv2.COLOR_BGR2RGB)
yolo_test_img1_rgb = cv2.cvtColor(yolo_test_img1, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18,10))

plt.subplot(1,2,1), plt.imshow(yolo_test_img0_rgb)
plt.subplot(1,2,2), plt.imshow(yolo_test_img1_rgb)

for ax in plt.gcf().axes:
    ax.axis("off")

#### $5)$ 모델 출력 결과 확인

YOLO 모델이 계산한 최종 BBox 좌표, 클래스 인덱스, 신뢰도 점수는 아래와 같이 확인할 수 있습니다.

In [ ]:
result = results[0]

print("* BBox 좌표:\n  ", result.boxes.xyxy)
print("\n* 클래스 Index:\n  ", result.boxes.cls)
print("\n* 객체별 Confidence Score:\n  ", result.boxes.conf)

#### $6)$ 클래스 지정 객체 탐지

YOLO 예측 함수인 `model.predict()`에는 원하는 클래스를 지정하여 객체를 탐지할 수 있는 기능이 있습니다.

In [ ]:
results = model.predict(
    source="src/images/city.png",
    conf=0.25,   # Confidence Threshold
    iou=0.5,     # IoU Threshold
    classes=[2]  # 2번 클래스 = 자동차
)

yolo_test_img = results[0].plot()  # 일반 .py 파일에서는 results[0].show()

yolo_test_img_rgb = cv2.cvtColor(yolo_test_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,8))
plt.imshow(yolo_test_img_rgb)
plt.axis("off");

---

### E. YOLO 실시간 객체 탐지

앞선 실습에서는 정적 이미지를 활용해 간단한 객체 탐지 과정을 경험해 보았습니다.

이번에는 CSI 카메라에서 들어오는 실시간 프레임에 YOLO 모델을 적용하여, 동적 영상 환경에서 객체를 실시간으로 탐지하는 실습을 진행해 보겠습니다.

기본적으로 ***Computer Vision*** 섹션에서 OpenCV를 활용한 실시간 카메라 피드 처리 방식과 동일합니다.

동일하게 GStreamer 파이프라인을 통해 OpenCV `VideoCapture`로 영상 스트림을 받아오도록 합시다.

```python
pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

while True:
    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    cv2.imshow("VideoCapture with GStreamer", frame)

cap.release()
cv2.destroyAllWindows()
```

GStreamer를 활용한 기본적인 실시간 카메라 프레임 처리 코드에 YOLO 모델을 불러와 실시간 객체 추적을 구현해봅시다.

지난 섹션에서 확인하셨다시피 Jupyter 환경에서는 OpenCV GUI 방식으로는 실행할 수 없는 관계로, 새로운 .py 파일에 복사하여 실행해봅시다.

```python
from ultralytics import YOLO
import cv2


model = YOLO("src/models/YOLO/yolo11n.pt")
model.to("cuda")

pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

while True:
    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = model.predict(
        source=frame,   # source image
        conf=0.25,      # Confidence Threshold
        iou=0.5,        # IoU Threshold
        verbose=False,  # no output prints
        classes=None,   # selected class
    )

    output_frame = results[0].plot()

    cv2.imshow("YOLO Object Detection", output_frame)

cap.release()
cv2.destroyAllWindows()
```

---

### F. 실시간 객체 탐지 성능 평가 (FPS)

YOLO 객체 탐지는 실시간성이 매우 뛰어난 모델이나, 하드웨어 성능, 모델의 크기, 입력 해상도 등, 많은 요소에 따라 처리 속도가 크게 변동됩니다.

특히 하드웨어 성능이 크게 제약되는 Edge 디바이스에서는 실시간 프레임 저하가 발생하기 쉽습니다.

따라서 목표하는 실시간 성능을 확보하기 위해 FPS를 정밀하게 측정하고 모니터링하여 성능을 최적화하는 과정이 필수적입니다.

FPS (Frames Per Second, 초당 프레임 수):
* 1 / 프레임_처리_시간_s  =  1000 / 프레임_처리_시간_ms
* 객체 탐지 모델이 초당 처리하는 이미지 프레임 수
* 실시간 처리의 핵심 지표
* 모델 크기(n, s, m, l, x)가 커질 수록 FPS 감소
* 정확도와의 Trade-off
* 엣지 디바이스(Edge Device)의 한계

객체 탐지에서의 FPS:
1. 추론 FPS
    * 모델 Forward 속도
2. End-to-End FPS
    * “프레임 입력 + 프레임 전처리 + 추론 + NMS + BBox 시각화 + 화면 출력” 속도
    * 체감 FPS
    * 실제 성능 평가

매 프레임 단위로 FPS를 즉시 계산하면 순간적인 연산 Bottleneck이나 프레임 드롭 때문에 수치의 변동 폭이 지나치게 커질 수 있습니다

이를 방지하고 안정적이고 부드러운 FPS 수치를 얻기 위해 지수 이동 평균(Exponential Moving Average, EMA) 방식을 활용합니다. EMA는 이전 프레임까지의 누적 FPS 평균값과 현재 측정된 FPS 값에 각각 가중치를 부여하여 결합함으로써, 최신 변경 사항을 반영하면서도 급격한 FPS 변화(Noise)를 완화해 줍니다.

위의 실시간 객체 탐지 코드에 FPS 계산 및 모니터링 코드를 추가하여 실시간성을 확인해봅시다.

```python
from ultralytics import YOLO
import cv2
import time


model = YOLO("src/models/YOLO/yolo11n.pt")
model.to("cuda")

pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

displayed_fps = 0.0

while True:
    start_time = time.perf_counter()

    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = model.predict(
        source=frame,   # source image
        conf=0.25,      # Confidence Threshold
        iou=0.5,        # IoU Threshold
        verbose=False,  # no output prints
        classes=None,   # selected class
    )

    output_frame = results[0].plot()

    elapsed_time = time.perf_counter() - start_time
    current_fps = 1.0 / elapsed_time

    if displayed_fps == 0:
        displayed_fps = current_fps
    else:
        displayed_fps = 0.9 * displayed_fps + 0.1 * current_fps

    cv2.putText(output_frame, f"FPS: {displayed_fps:.1f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    cv2.imshow("YOLO Object Detection with FPS", output_frame)

cap.release()
cv2.destroyAllWindows()
```

---

### G. Jetson 성능 모니터링

Jetson은 제한된 연산 자원과 메모리, 전력 및 발열 제약을 가진 엣지 디바이스입니다.

따라서 시스템을 효율적으로 최적화하기 위해 Jetson의 자원 사용 상태와 성능 병목을 실시간으로 확인할 필요가 있습니다.

다음 명령어로 실시간 성능 모니터링을 실시해봅시다.

```bash
tegrastats
```

혹은, Jetson 화면의 우측 상단에 Power Mode 메뉴에서 `Run tegrastats`를 선택하여 실행하는 방법도 있습니다.

처음 봤을 때에는 너무 많은 텍스트에 압도되어 어떤 정보도 쉽게 눈에 들어오지 않을 겁니다.

`tegrastats`의 출력 중 아래 키워드들을 먼저 이해하도록 합시다.

| 항목 | 의미 |
| --- | --- |
| `RAM` | 시스템 메모리 사용량 |
| `SWAP` | Swap 메모리 사용량 |
| `CPU` | 각 CPU 코어 사용률 및 동작 클럭 |
| `GR3D_FREQ` | GPU 사용률 및 GPU 클럭 |
| `EMC_FREQ` | 메모리 컨트롤러 사용률 및 클럭 |
| `cpu@...C` | CPU 온도 |
| `gpu@...C` | GPU 온도 |
| `VDD_IN` | Jetson 전체 전력 소비량 |

이 중에서 중요하게 확인해야 할 항목들은 아래와 같습니다.

```text
RAM            → 메모리 사용량
CPU            → CPU 사용률
GR3D_FREQ      → GPU 사용률
cpu@ / gpu@    → 온도
VDD_IN         → 전체 전력 소비량
```

`tetrastats` 출력 화면에서 필요한 키워드들을 직접 찾아가며 확인하는 것은 아무래도 직관성이 떨어지고 눈에 잘 들어오지 않을 겁니다.

따라서 아래 명령어를 실행하여 실시간 성능을 모니터링해 봅시다.

```bash
tegrastats | grep "RAM"
```

`grep` 명령어는 텍스트 데이터나 파일 내부에서 특정 키워드를 검색하여 강조하며 해당 줄만 추출해 주는 도구입니다.

복잡하고 길게 출력되는 tegrastats 정보 속에서 우리가 필요한 성능 지표만 선별하여 한눈에 확인하고 싶을 때 매우 유용하게 활용됩니다.

이제 YOLO 실시간 객체 탐지 코드를 실행하고, GPU에서 객체 탐지가 진행되는 동안 주요 성능 키워드들이 강조되도록 설정하여 실시간 모니터링을 진행해 봅시다.

---

### G. Power Mode

Jetson에는 CPU, GPU, 메모리 등의 최대 동작 범위를 조절하여 시스템의 성능 수준을 선택하는 Power Mode가 있습니다.

Jetson Orin Nano에는 기본적으로 다음과 같은 Power Mode가 제공됩니다.

| Mode ID | Power Mode | 전력 제한 | CPU 최대 클럭 | GPU 최대 클럭 | Memory 최대 클럭 | 성능 | 발열/전력 | 적합한 용도 |
|---:|---|---:|---:|---:|---:|---|---|---|
| **0** | **15W** | 15W | 1497.6 MHz | 612 MHz | 2133 MHz | 중간 | 중간 | 일반 개발, 가벼운 AI 추론 |
| **1** | **25W** | 25W | 1344 MHz | **918 MHz** | **3199 MHz** | 높음 | 높음 | YOLO 실시간 추론, 일반적인 고성능 AI |
| **2** | **MAXN SUPER** | 고정 전력 제한 없음 | **1728 MHz** | **1020 MHz** | **3199 MHz** | **최고** | **가장 높음** | 최대 FPS, TensorRT 벤치마크, 고성능 AI 추론 |

Power Mode를 고성능 옵션으로 설정하는 것이 항상 좋은 것은 아닙니다. Power Mode를 높이면 CPU와 GPU가 더 높은 성능으로 동작할 수 있지만, 그만큼 전력 소비와 발열도 증가합니다.

따라서 처리 속도는 향상될 수 있지만, 항상 일정한 최고 성능이 보장되는 것은 아닙니다. 시스템이 전력 또는 열 한계에 도달하면 Throttling이 발생하여 CPU/GPU 클럭이 제한되고 성능이 저하될 수 있습니다. 또한 전원 공급이 불안정하거나 온도가 위험 수준까지 상승하면 하드웨어 보호를 위해 Jetson이 강제로 Shutdown될 수 있습니다.

따라서 Power Mode는 무조건 가장 높은 모드를 사용하는 것이 아니라, 필요한 연산 성능, 전력 공급 능력, 배터리 사용 시간, 냉각 환경 등을 고려하여 적절하게 선택해야 합니다.

기본적으로 Power Mode는 Jetson 화면의 우측 상단에서 선택할 수 있습니다.

하지만 대부분의 경우, 모니터 없이 원격으로 제어하는 Headless 모드로 사용하기에 터미널 명령어로 설정해줍시다.

우선, 현재 Power Mode를 확인해볼까요?

```bash
sudo nvpmodel -q
```

위 명령어로 현재 Power Mode와 인덱스를 확인할 수 있습니다.

Jetson Orin Nano 기준, Power Mode 인덱스는 아래와 같습니다.

```text
0: 15W
1: 25W
2: MAXN
```

이제 Power Mode를 변경해봅시다.

`<mode_id>`를 원하는 Power Mode 인덱스로 바꿔 설정할 수 있습니다.

최대 성능을 활용할 수 있는 2번(MAXN)으로 설정해봅시다.

```bash
sudo nvpmodel -m <mode_id>
```

Power Mode를 바꿔가며 YOLO 실시간 객체 탐지 FPS를 비교해보도록 합시다.

---

### H. TensorRT 기반 YOLO 모델 최적화

YOLO 모델 최적화:
* 학습된 YOLO 모델을 배포 환경에서 더 빠르고 효율적으로 실행하기 위한 과정
* 객체 탐지 성능을 유지하면서 추론 시간과 자원 사용량을 줄이는 것이 목적

YOLO 모델 최적화 필요성:
* Jetson과 같은 Edge Device는 연산 성능, 메모리, 전력이 제한적
* 실시간 영상 처리에서는 추론 속도가 낮으면 FPS가 감소하고 응답 지연이 발생
* 실제 환경에서 안정적인 실시간 객체 탐지를 위해 모델 최적화가 중요

우선, PyTorch 기반 YOLO 모델(`.pt`)을 TensorRT 엔진(.engine)으로 변환하기 위해서는 `tensorrt` 라이브러리가 필요합니다.

변환 과정에서 Python 코드로 직접 `tensorrt` 라이브러리를 import하여 사용하지는 않지만, 모델 변환 과정에서 내부적으로 사용되므로 정상적으로 설치되어 있는지 확인하도록 합시다.

JetPack 6.2 환경에는 TensorRT 10.3이 기본적으로 포함되어 있습니다.

In [ ]:
import tensorrt
print(tensorrt.__version__)

PyTorch와 CUDA 또한 정상적으로 설치되어 있는지 다시 확인해봅시다.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch, TensorRT 패키지와 CUDA가 정상적으로 확인되었다면 다음 단계로 넘어가도록 합시다.

#### $1)$ ONNX 변환

ONNX:
* Open Neural Network Exchange
* 딥러닝 모델을 서로 다른 프레임워크에서 사용할 수 있도록 만든 표준 모델 형식
* PyTorch에 종속되지 않고 다양한 추론 환경에서 모델을 사용 가능
* TensorRT가 모델을 최적화할 수 있도록 연결해주는 중간 형식으로 활용


현재 모델:
```text
src/models/YOLO/yolo11n.pt
```
을 ONNX로 변환해봅시다.

우선, 기존 패키지들의 버전을 유지하면서 ONNX 변환에 필요한 패키지와 의존성들을 호환되는 버전으로 설치합시다.

```bash
pip install --no-deps onnx==1.16.0
pip install --no-deps onnxruntime==1.23.2
pip install --no-deps onnxslim==0.1.95
pip install --no-deps ml_dtypes==0.5.4
pip install --no-deps coloredlogs==15.0.1
pip install --no-deps humanfriendly==10.0
pip install --no-deps protobuf==3.20.3
```

혹시 모를 버전 충돌을 방지하기 위해 패키지 자동 설치 기능을 비활성화합니다.

In [ ]:
import os

os.environ["YOLO_AUTOINSTALL"] = "False"

In [ ]:
from ultralytics import YOLO


model = YOLO("src/models/YOLO/yolo11n.pt")

onnx_path = model.export(
    format="onnx",  # 모델을 ONNX 형식으로 내보내기
    imgsz=640,      # 입력 이미지 크기: 640×640
    batch=1,        # 한 번에 1장의 프레임 처리
    dynamic=False,  # 입력 크기를 640×640으로 고정 (Static Shape)
    simplify=True,  # ONNX 연산 그래프 단순화
    opset=20,       # ONNX 연산 규격 버전을 20으로 설정
)

print("ONNX:", onnx_path)

에러 없이 해당 코드가 완료되었다면, `src/models/YOLO/yolo11n.onnx` 파일이 생성되었음을 확인할 수 있습니다.

#### $2)$ 생성된 ONNX 검증

생성된 `yolo11n.onnx` 파일이 ONNX 형식으로 정상적으로 변환되었는지 확인해봅시다.

In [ ]:
import onnx


onnx_model = onnx.load(
    "src/models/YOLO/yolo11n.onnx"
)

onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")

입력 구조도 확인합시다.

일반적으로 YOLO export의 입력 Tensor 이름은 `images`입니다.

In [ ]:
for input_info in onnx_model.graph.input:
    print(input_info.name)

#### $3)$ TensorRT 변환

TensorRT:
* NVIDIA에서 제공하는 딥러닝 추론 최적화 라이브러리
* 학습이 끝난 모델을 NVIDIA GPU에서 더 빠르게 실행하도록 최적화
* 최적화된 모델은 보통 TensorRT Engine(.engine) 형태로 저장

TensorRT는 모델을 FP32, FP16, INT8 등의 정밀도로 최적화하여 TensorRT 엔진(`.engine`)으로 변환할 수 있습니다.

우선, TensorRT 모델 변환 및 성능 측정을 위한 CLI 도구인 `trtexec`을 사용하여 ONNX 모델을 FP32 TensorRT 엔진으로 변환하는 방법부터 살펴봅시다.

##### FP32 Engine으로 변환:

```bash
/usr/src/tensorrt/bin/trtexec \
    --onnx=$HOME/vision-llm/src/models/YOLO/yolo11n.onnx \
    --saveEngine=$HOME/vision-llm/src/models/YOLO/yolo11n_fp32.engine
```

PyTorch 모델은 기본적으로 FP32 정밀도를 사용합니다. 따라서 FP32 TensorRT 엔진으로 변환하면 연산 그래프 최적화, 연산 결합, GPU 실행 최적화 등은 적용되지만, 정밀도 감소를 통한 최적화는 적용되지 않습니다.

모델을 더욱 최적화하기 위해서는 FP32보다 FP16 정밀도를 사용하는 것이 좋습니다.

##### FP16 Engine으로 변환:

```bash
/usr/src/tensorrt/bin/trtexec \
    --onnx=$HOME/vision-llm/src/models/YOLO/yolo11n.onnx \
    --saveEngine=$HOME/vision-llm/src/models/YOLO/yolo11n_fp16.engine \
    --fp16
```

에러 없이 해당 명령어가 완료되었다면, `src/models/YOLO/yolo11n_fp16.engine` 파일이 생성되었음을 확인할 수 있습니다.

#### $4)$ 최적화 성능 확인

TensorRT 엔진으로 변환한 YOLO 모델의 추론 성능을 측정해봅시다.

아래 명령어를 사용하여 확인할 수 있습니다.

```bash
/usr/src/tensorrt/bin/trtexec \
    --loadEngine=$HOME/vision-llm/src/models/YOLO/yolo11n_fp16.engine \
    --warmUp=500 \
    --duration=10
```

출력 결과에서 확인해야 할 항목들은 다음과 같습니다.

| 항목 | 의미 |
| --- | --- |
| `Throughput` | 초당 처리 가능한 추론 횟수 |
| `Latency` | 한 번의 추론에 걸리는 전체 시간 |
| `GPU Compute Time` | GPU가 실제 모델 연산에 사용한 시간 |

#### $5)$ TensorRT 엔진으로 실시간 객체 탐지

이제 TensorRT 엔진으로 최적화 변환된 YOLO 모델을 활용하여 실시간 객체 탐지를 진행해 봅시다.

최적화 이전 모델과 FPS를 비교하며 성능 향상 폭을 확인해 보도록 합시다.

`.engine` 파일을 모델로 불러오는 경우에는, Engine 자체가 이미 NVIDIA GPU용으로 빌드된 파일이기 때문에 `model.to("cuda")`를 하지 않습니다.

```python
from ultralytics import YOLO
import cv2
import time


model = YOLO("src/models/YOLO/yolo11n_fp16.engine")

pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

displayed_fps = 0.0

while True:
    start_time = time.perf_counter()

    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = model.predict(
        source=frame,   # source image
        conf=0.25,      # Confidence Threshold
        iou=0.5,        # IoU Threshold
        verbose=False,  # no output prints
        classes=None,   # selected class
    )

    output_frame = results[0].plot()

    elapsed_time = time.perf_counter() - start_time
    current_fps = 1.0 / elapsed_time

    if displayed_fps == 0:
        displayed_fps = current_fps
    else:
        displayed_fps = 0.9 * displayed_fps + 0.1 * current_fps

    cv2.putText(output_frame, f"FPS: {displayed_fps:.1f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    cv2.imshow("YOLO Object Detection with FPS", output_frame)

cap.release()
cv2.destroyAllWindows()
```

---

### G. 양자화

양자화(Quantization)란?
* 모델의 가중치와 연산에 사용하는 숫자의 정밀도를 낮추는 최적화 기법
* 고정밀도 값을 더 적은 비트로 표현
* 장점: 모델 크기, 메모리 사용량 감소 및 추론 속도 향상
* 당점: 정확도 감소
* Edge Device에서 중요


Jetson은 연산 성능과 메모리와 같이 사용 가능한 리소스가 제한된 Edge Device이므로 모델을 더욱 효율적으로 최적화할 필요가 있습니다.

따라서 FP16보다 한 단계 더 낮은 정밀도인 INT8 양자화까지 적용해봅시다.

#### $1)$ Calibration 데이터셋 준비

INT8은 FP32보다 표현할 수 있는 값의 범위가 작기 때문에, FP32 값을 어떤 INT8 값으로 변환할지 기준을 정하는 과정이 필요합니다. 이를 Calibration이라고 합니다.

Calibration에서는 실제 추론 환경과 유사한 대표 이미지 데이터셋을 모델에 입력하여 각 Layer에서 발생하는 값의 분포를 분석하고, INT8 변환에 사용할 Quantization Scale을 계산합니다.

따라서 먼저 실제 사용 환경을 잘 대표할 수 있는 Calibration 데이터셋을 준비합니다.

Calibration 데이터셋은 일반적으로 수백 장 정도의 대표 이미지로 구성하며, 이번 실습에서는 약 500장의 이미지를 사용합니다.

실제 추론 환경과 최대한 유사한 데이터를 사용하기 위해 Jetson에 연결된 카메라의 프레임을 직접 캡처하여 Calibration 이미지로 저장합니다.

아래 코드를 별도 `.py` 파일에 복사하여 실행해봅시다.

```python
from pathlib import Path
import yaml

import cv2
from ultralytics import YOLO


CALIBRATION_DIR = Path("src/datasets/calibration")
IMAGE_DIR = CALIBRATION_DIR / "images"
LABEL_DIR = CALIBRATION_DIR / "labels"
YAML_PATH = CALIBRATION_DIR / "calibration.yaml"

NUM_IMAGES = 500
SAVE_EVERY_N_FRAMES = 5

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)


pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), "
    "width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)


frame_count = 0
save_count = 0

while save_count < NUM_IMAGES:
    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    display_frame = frame.copy()
    cv2.putText(display_frame, f"Calibration: {save_count}/{NUM_IMAGES}", (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
    cv2.imshow("Calibration Image Collection", display_frame)

    if frame_count % SAVE_EVERY_N_FRAMES == 0:
        # Image (jpg) 파일 저장
        image_path = IMAGE_DIR / f"{save_count:04d}.jpg"
        cv2.imwrite(str(image_path), frame)

        # Label (txt) 파일 저장 (빈 파일)
        label_path = LABEL_DIR / f"{save_count:04d}.txt"
        label_path.touch()

        save_count += 1

        print(f"Saved: {save_count}/{NUM_IMAGES}")

    frame_count += 1

cap.release()
cv2.destroyAllWindows()


if save_count < NUM_IMAGES:
    raise RuntimeError(f"이미지가 {save_count}장만 저장되었습니다.")


model = YOLO("src/models/YOLO/yolo11n.pt")

# YAML 파일에 저장할 Dictionary
calibration_yaml = {
    "path": str(CALIBRATION_DIR.resolve()),
    "train": "images",
    "val": "images",
    "names": model.names,
}

# YAML 파일 저장
with open(YAML_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        calibration_yaml,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print()
print("Calibration Dataset 생성 완료")
print(f"Images: {IMAGE_DIR}")
print(f"YAML:   {YAML_PATH}")
```

#### $2)$ INT8 양자화

앞에서 저장한 Calibration 데이터셋을 사용하여 모델의 값 분포를 분석하고, INT8 변환에 필요한 Quantization Scale을 계산합니다.

계산된 Scale을 기반으로 모델을 INT8 정밀도로 양자화하여 TensorRT Engine으로 변환합니다.

In [ ]:
from pathlib import Path


model = YOLO("src/models/YOLO/yolo11n.pt")

engine_path = model.export(
    format="engine",
    imgsz=640,
    quantize=8,
    data="src/datasets/calibration/calibration.yaml",
    batch=1,
    dynamic=False,
    device=0,
    nms=False,
)

print(f"TensorRT Engine 생성 완료: {engine_path}")

# 파일 이름 변경
int8_engine_path = Path(engine_path).replace("src/models/YOLO/yolo11n_int8.engine")

print(f"INT8 Engine: {int8_engine_path}")

에러 없이 해당 코드가 완료되었다면, `src/models/YOLO/yolo11n_int8.engine` 파일이 생성되었음을 확인할 수 있습니다.

#### $3)$ 최적화 성능 확인

INT8 양자화를 통해 TensorRT 엔진으로 변환한 YOLO 모델의 추론 성능을 측정해봅시다.

Ultralytics로 생성한 TensorRT Engine에는 모델 정보가 담긴 Ultralytics 전용 메타데이터가 앞부분에 추가됩니다.

하지만 trtexec은 순수 TensorRT Engine 형식만 읽기 때문에, 성능 측정 시에는 해당 메타데이터를 제거한 Raw Engine을 사용해야 합니다.

따라서 메타데이터를 제거한 INT8 엔진을 생성합시다.

In [ ]:
import json
from pathlib import Path


engine_path = Path("src/models/YOLO/yolo11n_int8.engine")
raw_engine_path = Path("src/models/YOLO/yolo11n_int8_raw.engine")

with engine_path.open("rb") as f:
    # 앞의 4 bytes: metadata 길이
    metadata_length = int.from_bytes(f.read(4), byteorder="little")

    # 메타데이터 읽기
    metadata = json.loads(f.read(metadata_length).decode("utf-8"))

    # 나머지 실제 TensorRT 엔진
    raw_engine = f.read()

raw_engine_path.write_bytes(raw_engine)

print(f"Metadata: {metadata}")
print(f"Raw TensorRT Engine: {raw_engine_path}")

이렇게 메타데이터를 제거하여 생성한 엔진 `yolo11n_int8_raw.engine`은 오직 `trtexec`으로 최적화 성능을 확인할 경우에만 사용합니다.

실제 객체 탐지에는 `yolo11n_int8_raw.engine`을 사용하지 않습니다.

이제 아래 명령어로 성능을 확인해봅시다.

```bash
/usr/src/tensorrt/bin/trtexec \
    --loadEngine=$HOME/vision-llm/src/models/YOLO/yolo11n_int8_raw.engine \
    --warmUp=500 \
    --duration=10
```

세 가지 최적화 모델들의 성능을 비교해봅시다.

```text
yolo11n_fp32.engine
yolo11n_fp16.engine
yolo11n_int8.engine
```

#### $4)$ INT8 TensorRT 엔진으로 실시간 객체 탐지

아래 코드를 `.py` 파일에 복사하여 INT8 TensorRT 엔진으로 실시간 객체 탐지를 시도해봅시다.

```python
from ultralytics import YOLO
import cv2
import time


model = YOLO("src/models/YOLO/yolo11n_int8.engine")

pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

displayed_fps = 0.0

while True:
    start_time = time.perf_counter()

    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = model.predict(
        source=frame,   # source image
        conf=0.25,      # Confidence Threshold
        iou=0.5,        # IoU Threshold
        verbose=False,  # no output prints
        classes=None,   # selected class
    )

    output_frame = results[0].plot()

    elapsed_time = time.perf_counter() - start_time
    current_fps = 1.0 / elapsed_time

    if displayed_fps == 0:
        displayed_fps = current_fps
    else:
        displayed_fps = 0.9 * displayed_fps + 0.1 * current_fps

    cv2.putText(output_frame, f"FPS: {displayed_fps:.1f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    cv2.imshow("YOLO Object Detection with FPS", output_frame)

cap.release()
cv2.destroyAllWindows()
```

INT8 TensorRT 엔진을 사용한 실시간 객체 탐지에서도 FP16 엔진과 동일하게 약 30 FPS가 출력됩니다.

이는 모델의 최대 추론 속도가 30 FPS이기 때문이 아니라, 카메라 Pipeline에서 입력 프레임 속도를 30 FPS로 설정했기 때문입니다.

INT8 TensorRT 엔진으로 객체 탐지가 정상적으로 수행되고 실시간으로 약 30 FPS가 유지된다면 최적화가 성공적으로 적용된 것입니다.

이것으로 **『딥러닝 기반 객체 탐지 시스템』** 섹션을 마무리합니다.

다음 섹션에서는 새로운 주제인 LLM에 관하여 배워보도록 하겠습니다.

---

## <center>< Section Project ></center>

본 섹션에서 배운 내용을 토대로 프로젝트를 진행합니다.

`src/videos/section4_project_traffic.mp4` 영상을 읽어 각 프레임의 신호등 상태가 빨강, 초록, 파랑 중 어떤 색인지 판단하고, 현재 신호 상태를 프레임 위에 텍스트로 표시해보도록 합시다.

In [ ]:
# TODO: Section 4 "DL Object Detection" Project

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026 김규래 (Kyu Rae Kim). All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>